# ResNet-18 Transfer Learning trên CIFAR-10

## Mục tiêu

Thực hiện và so sánh hai chiến lược Transfer Learning với ResNet-18 pretrained:

 **ResNet-18 A – FC only:** đóng băng backbone và chỉ huấn luyện lớp `fc`.
 
 **ResNet-18 B – Layer4 + FC:** fine-tuning `layer4` và `fc`.

Hai mô hình sử dụng cùng dữ liệu, cùng DataLoader và cùng cấu hình training để đảm bảo so sánh công bằng.

In [4]:
# ============================================================
# CELL 2 - IMPORT + PROJECT_ROOT
# ============================================================

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# 1. TÌM PROJECT_ROOT
# ============================================================

def find_project_root():
    """
    Tự tìm thư mục Practice_2 khi chạy trên:
    - VS Code / Windows
    - Google Colab
    """

    current = Path.cwd().resolve()

    # Kiểm tra thư mục hiện tại và các thư mục cha
    for path in [current, *current.parents]:

        # Trường hợp đang đứng ngay trong Practice_2
        if path.name == "Practice_2" and (path / "src").exists():
            return path

        # Trường hợp Practice_2 nằm bên trong thư mục hiện tại
        candidate = path / "Practice_2"

        if candidate.exists() and (candidate / "src").exists():
            return candidate.resolve()

    # Đường dẫn chuẩn khi chạy trên Google Colab
    colab_path = Path(
        "/content/UTH-Deep-Learning-nhom2/Practice_2"
    )

    if colab_path.exists() and (colab_path / "src").exists():
        return colab_path.resolve()

    raise FileNotFoundError(
        f"Không tìm thấy thư mục Practice_2.\n"
        f"Current directory: {current}"
    )


PROJECT_ROOT = find_project_root()


# ============================================================
# 2. THÊM PROJECT VÀO PYTHON PATH
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 3. IMPORT CODE DÙNG CHUNG CỦA NHÓM
# ============================================================

from src.data import (
    DataConfig,
    build_dataloaders,
)

from src.trainer import (
    get_device,
    train_model,
)

from src.models.resnet18 import (
    build_resnet18_fc_only,
    build_resnet18_layer4_fc,
)


# ============================================================
# 4. HÀM ĐẾM PARAMETERS
# ============================================================

def count_total_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def count_frozen_parameters(model):
    return (
        count_total_parameters(model)
        - count_trainable_parameters(model)
    )


# ============================================================
# 5. DEVICE
# ============================================================

device = get_device()


# ============================================================
# 6. KIỂM TRA PROJECT
# ============================================================

print("===== PROJECT SETUP =====")
print("Current dir  :", Path.cwd())
print("PROJECT_ROOT :", PROJECT_ROOT)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())
print("Device       :", device)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

===== PROJECT SETUP =====
Current dir  : d:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\notebooks
PROJECT_ROOT : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2
PyTorch      : 2.13.0+cpu
CUDA         : False
Device       : cpu


In [9]:
import inspect

print("FC ONLY:")
print(inspect.signature(build_resnet18_fc_only))

print("\nLAYER4 + FC:")
print(inspect.signature(build_resnet18_layer4_fc))

FC ONLY:
(num_classes=10)

LAYER4 + FC:
(num_classes=10)


In [6]:
# ============================================================
# CELL 3 - DATALOADER CHUNG
# ============================================================

data_config = DataConfig(
    data_dir=PROJECT_ROOT / "data" / "raw",
    batch_size=32,
    num_workers=0,
)

data_bundle = build_dataloaders(data_config)

train_loader = data_bundle["train_loader"]
val_loader = data_bundle["val_loader"]
test_loader = data_bundle["test_loader"]
class_names = data_bundle["class_names"]

print("===== DATALOADER READY =====")
print("Data directory :", data_config.data_dir)

print("Train samples  :", len(train_loader.dataset))
print("Val samples    :", len(val_loader.dataset))
print("Test samples   :", len(test_loader.dataset))

print("Train batches  :", len(train_loader))
print("Val batches    :", len(val_loader))
print("Test batches   :", len(test_loader))

print("Classes        :", class_names)
print("Number classes :", len(class_names))

===== DATALOADER READY =====
Data directory : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\data\raw
Train samples  : 45000
Val samples    : 5000
Test samples   : 10000
Train batches  : 1407
Val batches    : 157
Test batches   : 313
Classes        : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Number classes : 10


In [10]:
# ============================================================
# CELL 4 - XÂY RESNET-18 A VÀ B
# ============================================================

num_classes = len(class_names)

# ResNet-18 A:
# Chỉ huấn luyện lớp FC
model_a = build_resnet18_fc_only(
    num_classes=num_classes
)

# ResNet-18 B:
# Fine-tuning layer4 + FC
model_b = build_resnet18_layer4_fc(
    num_classes=num_classes
)

# Đưa model về đúng device hiện tại
model_a = model_a.to(device)
model_b = model_b.to(device)

print("===== MODELS READY =====")
print("Model A : ResNet-18 - FC only")
print("Model B : ResNet-18 - Layer4 + FC")
print("Classes :", num_classes)
print("Device  :", device)

===== MODELS READY =====
Model A : ResNet-18 - FC only
Model B : ResNet-18 - Layer4 + FC
Classes : 10
Device  : cpu


In [11]:
# ============================================================
# CELL 5 - SO SÁNH TRAINABLE PARAMETERS
# ============================================================

total_params_a = count_total_parameters(model_a)
trainable_params_a = count_trainable_parameters(model_a)
frozen_params_a = count_frozen_parameters(model_a)

total_params_b = count_total_parameters(model_b)
trainable_params_b = count_trainable_parameters(model_b)
frozen_params_b = count_frozen_parameters(model_b)

print("===== PARAMETER COMPARISON =====")

print("\nResNet-18 A - FC only")
print("Total params     :", f"{total_params_a:,}")
print("Trainable params :", f"{trainable_params_a:,}")
print("Frozen params    :", f"{frozen_params_a:,}")

print("\nResNet-18 B - Layer4 + FC")
print("Total params     :", f"{total_params_b:,}")
print("Trainable params :", f"{trainable_params_b:,}")
print("Frozen params    :", f"{frozen_params_b:,}")

===== PARAMETER COMPARISON =====

ResNet-18 A - FC only
Total params     : 11,181,642
Trainable params : 5,130
Frozen params    : 11,176,512

ResNet-18 B - Layer4 + FC
Total params     : 11,181,642
Trainable params : 8,398,858
Frozen params    : 2,782,784


In [12]:
# ============================================================
# CELL 6 - FORWARD PASS CHECK
# ============================================================

# Lấy 1 batch từ tập train
images, labels = next(iter(train_loader))

# Đưa dữ liệu lên cùng device với model
images = images.to(device)
labels = labels.to(device)

# Chuyển sang eval để kiểm tra forward pass
model_a.eval()
model_b.eval()

# Không tính gradient vì chỉ kiểm tra
with torch.no_grad():
    outputs_a = model_a(images)
    outputs_b = model_b(images)

print("===== FORWARD PASS CHECK =====")
print("Device         :", device)
print("Images device  :", images.device)
print("Labels device  :", labels.device)

print("\nInput shape    :", images.shape)
print("Labels shape   :", labels.shape)
print("Output A shape :", outputs_a.shape)
print("Output B shape :", outputs_b.shape)

===== FORWARD PASS CHECK =====
Device         : cpu
Images device  : cpu
Labels device  : cpu

Input shape    : torch.Size([32, 3, 224, 224])
Labels shape   : torch.Size([32])
Output A shape : torch.Size([32, 10])
Output B shape : torch.Size([32, 10])


In [13]:
# ============================================================
# CELL 7 - CẤU HÌNH TRAINING CHUNG
# ============================================================

# Số epoch
NUM_EPOCHS = 5

# Learning rate
LEARNING_RATE = 0.001

# Loss function dùng chung
criterion = nn.CrossEntropyLoss()

# Chỉ đưa các parameter có requires_grad=True vào optimizer
optimizer_a = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_a.parameters()),
    lr=LEARNING_RATE
)

optimizer_b = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_b.parameters()),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# Thư mục checkpoint
# ------------------------------------------------------------

checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_a = checkpoint_dir / "resnet18_fc_only_best.pth"
checkpoint_b = checkpoint_dir / "resnet18_layer4_fc_best.pth"

# ------------------------------------------------------------
# Hiển thị cấu hình
# ------------------------------------------------------------

print("===== TRAINING CONFIG =====")
print("Epochs           :", NUM_EPOCHS)
print("Learning rate    :", LEARNING_RATE)
print("Batch size       :", data_config.batch_size)
print("Loss function    : CrossEntropyLoss")
print("Optimizer A      : Adam")
print("Optimizer B      : Adam")
print("Device           :", device)

print("\nCheckpoint A     :", checkpoint_a)
print("Checkpoint B     :", checkpoint_b)

===== TRAINING CONFIG =====
Epochs           : 5
Learning rate    : 0.001
Batch size       : 32
Loss function    : CrossEntropyLoss
Optimizer A      : Adam
Optimizer B      : Adam
Device           : cpu

Checkpoint A     : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\checkpoints\resnet18_fc_only_best.pth
Checkpoint B     : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\checkpoints\resnet18_layer4_fc_best.pth
